# Training Series: Introduction to eXtended Discontinuous Galerkin Methods for Multi-Phase Flow Problems <br> - Hands-On Worksheets

In [ ]:
#r "../BoSSSpad/BoSSSpad.dll"
//#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net5.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Hands-On 3: Poisson Equation and the Symmetric Interior Penalty Flux 

## Symmteric Interior Penalty

### Problem statement

We consider the 2D Poisson problem:
$$ \Delta u = f(x,y) $$

where $f(x,y) \neq 0$ is an arbitrary function of $x$ and/or $y$.
Within this exercise, we are going to investigate the Symmetric Interior Penalty discretization method (SIP) for the Laplace operator

$$a_{\text{sip}}(u,v)
= \int_{\Omega} \underbrace{\nabla u \cdot \nabla v}_{\text{Volume\ term}}dV
  - \oint_{\Gamma \setminus \Gamma_{N }} \underbrace{
        \{\nabla u\} \cdot n_{\Gamma} \llbracket v \rrbracket
     }_{\text{consistency term}} + \underbrace{
        \{\nabla v\} \cdot \vec{n}_{\Gamma} \llbracket u \rrbracket
     }_{\text{symmetry term}} dA
  + \oint_{\Gamma \setminus \Gamma_{N}} \underbrace{
       \eta \llbracket u \rrbracket \llbracket v \rrbracket
    }_{\text{penalty term}} dA$$

The use of these fluxes including a penalty term stabilizes the DG-discretization for the Laplace operator.

### Implementation of the SIP fluxes

We need the following packages:

In [ ]:
using ilPSP.LinSolvers;
using NUnit.Framework;

First, we need a class in which the integrands are defined.
This also includes some technical aspects like the *TermActivationFlags*.

In [ ]:
public class SipLaplace :
        BoSSS.Foundation.IEdgeForm,   // edge integrals
        BoSSS.Foundation.IVolumeForm, // volume integrals
        IEquationComponentCoefficient // update of coefficients (e.g. length scales) required for penalty parameters 
{
    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u" }; } 
    }
    /// The \code{TermActivationFlags} tell \BoSSS which kind of terms, 
    /// i.e. products of u, v, \nabla u, and \nabla v
    /// the VolumeForm(...) actually contains.
    /// This additional information helps to improve the performance.
    public TermActivationFlags VolTerms {
        get {
            return TermActivationFlags.GradUxGradV;
        }
    }
    /// activation flags for the 'InnerEdgeForm(...)'
    public TermActivationFlags InnerEdgeTerms {
        get {
            return (TermActivationFlags.AllOn);
            // if we do not care about performance, we can activate all terms.
        }
    }
    public TermActivationFlags BoundaryEdgeTerms {
       get {
           return TermActivationFlags.AllOn;
        }
    }

    /// For the computation of the penalty factor $\eta$,
    /// we require some length scale for each cell and 
    /// the polynomial degree of the DG approximation.
    MultidimensionalArray cj;

    double penalty_base; // base factor must be scaled by polynomial degree  

    /// The additional scaling of the penalty by polynomial degree 
    /// and in depencence of geometry can be obtained through 
    /// impmenting the \code{IEquationComponentCoefficient} interface:
    public void CoefficientUpdate(CoefficientSet cs, int[] DomainDGdeg, int TestDGdeg) {
        int D = cs.GrdDat.SpatialDimension;
        double _D = D;
        double _p = DomainDGdeg.Max();

        double penalty_deg_tri = (_p + 1) * (_p + _D) / _D; // formula for triangles/tetras
        double penalty_deg_sqr = (_p + 1.0) * (_p + 1.0); // formula for squares/cubes

        penalty_base = Math.Max(penalty_deg_tri, penalty_deg_sqr); // the conservative choice
        //Console.WriteLine("Setting penalty base factor for deg " + _p + " to " + penalty_base);

        cj = ((GridData)(cs.GrdDat)).Cells.cj; // inverse length scale (dimension is one-over-length, resp. area over volume)
    }     
    
            
    /// The safety factor for the penalty factor should be in the order of 1.
    /// A very large penalty factor increases the condition number of the system, but without affecting stability.
    /// A very small penalty factor yields to an unstable discretization.
    public double PenaltySafety = 2.2; 

    /// The actual computation of the penalty factor, which should be used in the \code{InnerEdgeForm} and \code{BoundaryEdgeForm} functions.
    /// Hint: for the parameters \code{jCellIn}, \code{jCellOut} and \code{g}, take a look at \code{CommonParams} and \code{CommonParamsBnd}.
    double PenaltyFactor(int jCellIn, int jCellOut) {
        double cj_in         = cj[jCellIn];
        double eta           = penalty_base * cj_in * PenaltySafety;
        if(jCellOut >= 0) {
            double cj_out = cj[jCellOut];
            eta           = Math.Max(eta, penalty_base * cj_out * PenaltySafety);
        }
        
        return eta;
    }
    
    /// The following functions cover the actual math.
    /// For any discretization of the Laplace operator, we have to specify:
    /// 
    /// - a volume integrand,
    /// - an edge integrand for inner edges, i.e. on $\Gamma_i$,
    /// - an edge integrand for boundary edges, i.e. on $\partial \Omega$.
    /// 
    /// The integrand for the volume integral:
    public double VolumeForm(ref CommonParamsVol cpv, 
           double[] U, double[,] GradU, // the trial-function u (i.e. the function we search for) and its gradient
           double V, double[] GradV     // the test function; 
           ) {
 
        // == TODO == add the SIP volume form 
        double acc = 0.0;

        return acc;
    }

    /// The integrand for the integral on the inner edges,
    /// 
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   + \eta [[u]] [[v]] :
    /// 
    public double InnerEdgeForm(ref CommonParams inp, 
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT, 
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) {
 
        double eta = PenaltyFactor(inp.jCellIn, inp.jCellOut);
 
        double Acc = 0.0;
        // == TODO == add the consistency term
        // == TODO == add the cymmetry term
        // == TODO == add the penalty term

        return Acc;
    } 

    /// The integrand on boundary edges, i.e. on $\partial \Omega$, is
    ///  
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   +  \eta [[u]]  [[v]] .
    /// 
    /// For the boundary we have to consider the special definition for 
    /// the mean-value operator ${-}$ and the jump operator $[[-]]$ on the boundary.
    public double BoundaryEdgeForm(ref CommonParamsBnd inp, 
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_IN) {
 
        double eta = PenaltyFactor(inp.jCellIn, -1);
        
        double Acc = 0.0;
        // == TODO == add the consistency term
        // == TODO == add the cymmetry term
        // == TODO == add the penalty term
 
        return Acc;
    }
}

### Evaluation of the Poisson operator in 1D

We consider the following problem:
$$
\Delta u = 2,\quad -1<x<1,\quad u(-1)=u(1)=0.
$$
The solution is $u_{ex}(x) = 1 - x^2$. Since this is quadratic, we can represent it **exactly** in a DG space of order 2.
As usual, we have to set up a grid, a basis and a right-hand-side.

In [ ]:
var grd1D                     = Grid1D.LineGrid(GenericBlas.Linspace(-1,1,10));
var DGBasisOn1D               = new Basis(grd1D, 2);
var RHS                       = new SinglePhaseField(DGBasisOn1D, "RHS");
RHS.ProjectField((double x) => 2);

In [ ]:
var i_SipLaplace              = new SipLaplace();
var Operator_SipLaplace       = i_SipLaplace.Operator();

We now want to calculate the residual after inserting the exact solution as well as a wrong solution. 
The implementation of the exact solution:

In [ ]:
var u_ex         = new SinglePhaseField(DGBasisOn1D, "$u_{ex}$");
u_ex.ProjectField((double x) => 1.0 - x*x);

The implementation of a spurious, i.e. a wrong solution; we take the exact solution and add random values in each cell:

In [ ]:
var u_wrong      = new SinglePhaseField(DGBasisOn1D, "$u_{wrong}$");
u_wrong.ProjectField((double x) => 1.0 - x*x);
Random R         = new Random();
for(int j = 0; j < grd1D.GridData.Cells.NoOfLocalUpdatedCells; j++){
    double ujMean = u_wrong.GetMeanValue(j);
    ujMean += R.NextDouble();
    u_wrong.SetMeanValue(j, ujMean);
}

Evaluating the Laplace operator using the different solutions:

In [ ]:
var Residual     = new SinglePhaseField(DGBasisOn1D,"Resi1");
var ResidualNorm = new List<double>();
foreach(var u in new DGField[] {u_ex, u_wrong}) {
    Residual.Clear();
    Operator_SipLaplace.Evaluate(u, Residual);  // evaluate
    Residual.Acc(-1.0, RHS);    
    double ResiNorm = Residual.L2Norm();
    ResidualNorm.Add(ResiNorm);
    Console.WriteLine("Residual for " + u.Identification + " = " + ResiNorm);  
}

In [ ]:
Assert.LessOrEqual(ResidualNorm[0], 1e-10);
Assert.GreaterOrEqual(ResidualNorm[1], 1e-1);

### The penalty parameter of the SIP and stability in 2D

We define a two-dimensional grid:

In [ ]:
var grd2D       = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,21), 
                                         GenericBlas.Linspace(-1,1,16));
var DGBasisOn2D = new Basis(grd2D, 5);
var Mapping2D   = new UnsetteledCoordinateMapping(DGBasisOn2D);

We consider the example 
$$
    -\Delta u = \pi^2 (a_x^2 + a_y^2)/4 \cos(a_x \pi x/2) \cos(a_y \pi y/2) 
      \text{ with } 
      (x,y) \in (-1,1)^2
$$
and $u = 0$ on the boundary.
The exact solution is $u_{Ex}(x,y) = \cos(a_x \pi  x/2) \cos(a_x \pi y/2)$, where $a_x$ and $a_y$ must be odd numbers
to comply with homogeneous bounary condition.

In [ ]:
double ax = 1.0; // must be an odd number to comply with homogeneous Dirichlet boundary condition
double ay = 3.0; // must be an odd number to comply with homogeneous Dirichlet boundary condition
Func<double[], double> exSol = 
        (X => Math.Cos(X[0]*ax*Math.PI*0.5)*Math.Cos(X[1]*ay*Math.PI*0.5));
Func<double[], double> exRhs = 
        (X => ((ax.Pow2() + ay.Pow2())/4.0)*Math.PI.Pow2()
             *Math.Cos( X[0]*ax*Math.PI*0.5 )*Math.Cos( X[1]*ay*Math.PI*0.5 )); // == - /\ exSol

SinglePhaseField RHS = new SinglePhaseField(DGBasisOn2D, "RHS");
RHS.ProjectField(exRhs);
double[] RHSvec = RHS.CoordinateVector.ToArray();

We check our discretization once more in 2D; the residual should be low,
but not exactly (resp. up to $10^{-12}$) since the solution is not 
polynomial and cannot be fulfilled exactly.

In [ ]:
SinglePhaseField u = new SinglePhaseField(DGBasisOn2D,"u");
u.ProjectField(exSol);

var Matrix_SIP_sf = Operator_SipLaplace.ComputeMatrix(Mapping2D, null, Mapping2D);

SinglePhaseField Residual = new SinglePhaseField(DGBasisOn2D,"Residual");
Residual.Acc(1.0, RHS);
Matrix_SIP_sf.SpMV(-1.0, u.CoordinateVector, 1.0, Residual.CoordinateVector);
Console.WriteLine("Residual L2 norm: " + Residual.L2Norm());

== Task == How could you decrease the Residual L2 norm?

We also check that the matrix is symmetric:

In [ ]:
var checkMatrix = Matrix_SIP_sf - Matrix_SIP_sf.Transpose();
checkMatrix.InfNorm()

In [ ]:
Assert.LessOrEqual(checkMatrix.InfNorm(), 1e-8);

### Convergence study, indefinite vs. definite.

We are going to solve the SIP-system for different grid resolutions,
comparing an insufficient penalty (*PenaltySafety*-factor $\eta_0 = 0.01$) to a penalty which is large enough ($\eta_0 = 2$).

In [ ]:
double[] Resolution = new double[] { 2, 4, 8, 16, 32, 64 };
List<double> L2Error_indef  = new List<double>();
List<double> L2Error_posdef = new List<double>();
int cnt = 0;
foreach(int Res in Resolution) {
    cnt++;
    //var grd2D = Grid2D.UnstructuredTriangleGrid(GenericBlas.Linspace(-1,1,(int)Res + 1), 
    //                                            GenericBlas.Linspace(-1,1,(int)Res + 1));
    var grd2D = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,(int)Res + 1), 
                                      GenericBlas.Linspace(-1,1,(int)Res + 1));
    var gdata2D = new GridData(grd2D);
    var DGBasisOn2D = new Basis(gdata2D, 2);
    var Mapping2D  = new UnsetteledCoordinateMapping(DGBasisOn2D);
 
    SinglePhaseField RHS = new SinglePhaseField(DGBasisOn2D, "RHS");
    RHS.ProjectField(exRhs);
    SinglePhaseField uEx = new SinglePhaseField(
           new Basis(gdata2D, DGBasisOn2D.Degree*2),
           "Error");
    uEx.ProjectField(exSol);
 
 
    i_SipLaplace.PenaltySafety    = -1; // == TODO == choose a insufficient small penalty parameter (indefinite matrix)
    var Matrix_SIP_indef          = Operator_SipLaplace.ComputeMatrix(
                                    Mapping2D,null,Mapping2D);
 
    SinglePhaseField u_indef = new SinglePhaseField(DGBasisOn2D,"u_indef");
    Matrix_SIP_indef.Solve_Direct(u_indef.CoordinateVector, 
                                  RHS.CoordinateVector);
    var Error_indef = uEx.CloneAs();
    Error_indef.AccLaidBack(-1.0, u_indef);
    L2Error_indef.Add(Error_indef.L2Norm());
 

    i_SipLaplace.PenaltySafety    = -1; // == TODO == choose a sufficient large penalty parameter (definite matrix)
    var Matrix_SIP_posdef         = Operator_SipLaplace.ComputeMatrix(
                                    Mapping2D, null, Mapping2D);
 
    SinglePhaseField u_posdef = new SinglePhaseField(DGBasisOn2D,"u_posdef");
    Matrix_SIP_posdef.Solve_Direct(u_posdef.CoordinateVector, 
                                   RHS.CoordinateVector);
    var Error_posdef = uEx.CloneAs();
    Error_posdef.AccLaidBack(-1.0, u_posdef);
    L2Error_posdef.Add(Error_posdef.L2Norm());
    
    //Tecplot("ConvStudy-" + cnt, uEx, u_posdef, u_indef); // activate this line for plotting!
    
    Console.WriteLine(L2Error_indef.Last().ToString("0.#E-00") 
                      + "\t" + L2Error_posdef.Last().ToString("0.#E-00"));
}

### Convergence Plot and Conclusions

The convergence plot should unveil that there is something wrong if the
penalty factor is set too low.
Unfortunately, **it does not**, so this is some kind of **anti-example**;
It is in this tutorial anyway **to illustrate the difficulties of numerical testing**.

The reason why the indefinite matrix still gives a solution convergence 
is very likely that the solver which is used in BoSSS is also (sometimes) capable
of solving singular or close-to-singular systems, i.e. systems without a unique solution.
In those cases, it selects a solution with a minimal solution norm. 
Since BoSSS uses an orthonormal basis the $L^2$ norm of the DG-Field is identical to the $l_2$ norm of the 
coordinate vector (Parseval's identity).
Therefore, the solver by chance adds additional stability which is not part of the (instable) discretization.

The error of the positive definite system, *Error\_posdef*, where the 
penalty is chosen sufficiently large converges with the expected 
rate.

In [ ]:
var plt = new Plot2Ddata();
plt.AddDataGroup("indef mtx", Resolution, L2Error_indef);
plt.AddDataGroup("pos def mtx", Resolution, L2Error_posdef);

/// A double-logarithmic scale is used:
plt.LogX = true;
plt.LogY = true;

/// Set Format
plt.dataGroups[0].Format =  new PlotFormat(lineColor: LineColors.Red, 
                                           pointSize: 2, 
                                           dashType: DashTypes.DotDashed, 
                                           Style: Styles.LinesPoints, 
                                           pointType:PointTypes.OpenCircle);
plt.dataGroups[1].Format =  new PlotFormat("Blue-.o"); // altenatively, using MATLAB-like format strings
plt.dataGroups[1].Format.PointSize = 2;
// Show!
plt.PlotNow()

Finally, we are going to compute the convergence rate of the SIP
discretization.

We compute the slope of the log-log plot:

In [ ]:
double dk = Resolution.LogLogRegressionSlope(L2Error_posdef);
dk

In [ ]:
Assert.LessOrEqual(dk, -2.9);